In [7]:
import pandas as pd
import numpy as np
import datetime

# 加载数据
data = pd.read_excel('data\match1701.xlsx')

def convert_to_seconds(time_val):
    if isinstance(time_val, str):
        # 如果是字符串，按原来的方式处理
        h, m, s = map(int, time_val.split(':'))
        return h * 3600 + m * 60 + s
    elif isinstance(time_val, pd._libs.tslibs.timedeltas.Timedelta):
        # 如果time_val是Timedelta对象（例如，从pd.to_timedelta()得到的）
        return time_val.total_seconds()
    elif isinstance(time_val, datetime.time):
        # 如果time_val是datetime.time对象
        return time_val.hour * 3600 + time_val.minute * 60 + time_val.second
    else:
        # 如果都不是，返回None或抛出一个错误
        return None

data['total_seconds'] = data['elapsed_time'].apply(convert_to_seconds)
# 计算比赛的总时长
total_duration = data['total_seconds'].max()

# 定义时间窗口的边界
time_windows = [0.2, 0.4, 0.6, 0.8, 1.0]

# 初始化一个字典来存储结果
results = {'Window': [], 'Player 1 Points Won': [], 'Player 2 Points Won': []}

# 计算每个时间窗口内的得分
for i, window in enumerate(time_windows):
    # 计算当前窗口的结束时间
    end_time = total_duration * window
    
    # 选取当前窗口内的数据
    window_data = data[data['total_seconds'] <= end_time]
    
    # 由于数据可能是累积的，我们需要计算增量
    if i == 0:
        p1_points = window_data['p1_points_won'].sum()
        p2_points = window_data['p2_points_won'].sum()
    else:
        prev_end_time = total_duration * time_windows[i-1]
        prev_window_data = data[data['total_seconds'] <= prev_end_time]
        p1_points = window_data['p1_points_won'].sum() - prev_window_data['p1_points_won'].sum()
        p2_points = window_data['p2_points_won'].sum() - prev_window_data['p2_points_won'].sum()
    
    # 存储结果
    results['Window'].append(f"{int(window*100)}%")
    results['Player 1 Points Won'].append(p1_points)
    results['Player 2 Points Won'].append(p2_points)

# 将结果转换为DataFrame以便查看
results_df = pd.DataFrame(results)
print(results_df)

  Window  Player 1 Points Won  Player 2 Points Won
0    20%                  933                 1413
1    40%                 2969                 3596
2    60%                 5058                 5337
3    80%                 7587                 7298
4   100%                10849                10905
